# 🧪 Data Extraction Evaluation Pipeline: Submission vs. Ground Truth

This script aligns model predictions with ground truth data for the table extraction task.  It evaluates the extracted standard markdown tables using TEDS (Tree Edit Distance Similarity) and RMS (Relative Mapping Similarity).

In [ ]:
import json
import numpy as np
from pathlib import Path
import re
from scipy.optimize import linear_sum_assignment
from typing import List, Tuple, Union
import distance
from lxml import html
from apted import APTED, Config
from apted.helpers import Tree
from tqdm.auto import tqdm

In [ ]:
BASE_DIR = Path.cwd().parent
EVAL_CATEGORY = "dev"
EVAL_DATA_DIR = (
    BASE_DIR / "ALD-E-ImageMiner" / "icdar2026-competition-data" / EVAL_CATEGORY
)

SUBMISSION_FILE = BASE_DIR / "submission_finetuning_extraction_dev.json"
RESULTS_FILE = BASE_DIR / "evaluation_extraction_results.json"

In [ ]:
def normalized_levenshtein(s1: str, s2: str) -> float:
    """Computes Normalized Levenshtein Distance between two strings."""
    if not s1 and not s2:
        return 0.0
    m, n = len(s1), len(s2)
    dp = [[0] * (n + 1) for _ in range(m + 1)]
    for i in range(m + 1):
        dp[i][0] = i
    for j in range(n + 1):
        dp[0][j] = j
    for i in range(1, m + 1):
        for j in range(1, n + 1):
            cost = 0 if s1[i - 1] == s2[j - 1] else 1
            dp[i][j] = min(dp[i - 1][j] + 1, dp[i][j - 1] + 1, dp[i - 1][j - 1] + cost)
    max_len = max(m, n)
    return dp[m][n] / max_len if max_len > 0 else 0.0


def safe_float(val: str) -> Union[float, str]:
    """Attempts to parse a string into a float, stripping characters like $ or %."""
    if isinstance(val, (int, float)):
        return float(val)
    cleaned = re.sub(r"[^\d\.\-eE]", "", str(val))
    try:
        return float(cleaned)
    except ValueError:
        return str(val).strip()


def numeric_relative_distance(v_pred, v_target, theta: float) -> float:
    """Computes bounded relative distance between numeric entries."""
    v_p = safe_float(v_pred)
    v_t = safe_float(v_target)

    if isinstance(v_p, str) or isinstance(v_t, str):
        dist = normalized_levenshtein(str(v_p), str(v_t))
        return 1.0 if dist > theta else dist

    if v_t == 0.0:
        dist = 0.0 if v_p == 0.0 else 1.0
    else:
        dist = abs(v_p - v_t) / abs(v_t)
    return min(1.0, dist) if dist <= theta else 1.0


def markdown_to_mapping(md_string: str) -> List[Tuple[str, str, str]]:
    """
    Parses a markdown table into a list of (row_header, col_header, value) mappings.
    Assumes standard format: first column is the row header, first row is column headers.
    """
    if not md_string or not isinstance(md_string, str):
        return []

    lines = [line.strip() for line in md_string.strip().split("\n") if line.strip()]
    if len(lines) < 3:
        return []

    # Extract column headers
    col_headers = [c.strip() for c in lines[0].strip("|").split("|")]

    mappings = []
    # Skip line 0 (headers) and line 1 (separators like |---|---|)
    for line in lines[2:]:
        cells = [c.strip() for c in line.strip("|").split("|")]
        if not cells:
            continue

        row_header = cells[0]
        # Map remaining cells to their corresponding column headers
        for col_idx in range(1, len(cells)):
            col_header = (
                col_headers[col_idx] if col_idx < len(col_headers) else f"Col_{col_idx}"
            )
            value = cells[col_idx]
            mappings.append((row_header, col_header, value))

    return mappings


def compute_rms_directional(
    P: List[Tuple[str, str, str]],
    T: List[Tuple[str, str, str]],
    tau: float,
    theta: float,
) -> Tuple[float, float, float]:
    N, M = len(P), len(T)
    if N == 0 and M == 0:
        return 1.0, 1.0, 1.0
    if N == 0 or M == 0:
        return 0.0, 0.0, 0.0

    cost_matrix = np.zeros((N, M))
    for i, (pr, pc, _) in enumerate(P):
        p_key = f"{pr}||{pc}"
        for j, (tr, tc, _) in enumerate(T):
            t_key = f"{tr}||{tc}"
            nl_dist = normalized_levenshtein(p_key, t_key)
            cost_matrix[i, j] = 1.0 if nl_dist > tau else nl_dist

    row_ind, col_ind = linear_sum_assignment(cost_matrix)

    total_similarity = 0.0
    for i, j in zip(row_ind, col_ind):
        key_dist = cost_matrix[i, j]
        _, _, p_val = P[i]
        _, _, t_val = T[j]
        val_dist = numeric_relative_distance(p_val, t_val, theta)

        similarity = (1.0 - key_dist) * (1.0 - val_dist)
        total_similarity += similarity

    precision = total_similarity / N
    recall = total_similarity / M
    f1 = (
        2 * (precision * recall) / (precision + recall)
        if (precision + recall) > 0
        else 0.0
    )

    return precision, recall, f1


# --- 4. Main Pipeline Hook ---


def calculate_rms(
    pred_md: str, gt_md: str, tau: float = 0.5, theta: float = 0.1
) -> float:
    """
    Main entry point for the pipeline. Parses markdown and computes highest F1 RMS score
    accounting for potential table transpositions.
    """
    P = markdown_to_mapping(pred_md)
    T = markdown_to_mapping(gt_md)

    if not P and not T:
        return 1.0
    if not P or not T:
        return 0.0

    # Base Evaluation
    _, _, f1_base = compute_rms_directional(P, T, tau, theta)

    # Transposed Evaluation (swap row and col headers in P)
    P_transposed = [(pc, pr, pv) for pr, pc, pv in P]
    _, _, f1_trans = compute_rms_directional(P_transposed, T, tau, theta)

    return max(f1_base, f1_trans)

In [ ]:
def markdown_to_html(md_string: str) -> str:
    """
    Converts a standard markdown table into an HTML table string.
    This is required because TEDS evaluates the structural tree of tables.
    """
    if not md_string or not isinstance(md_string, str):
        return "<html><body><table></table></body></html>"

    lines = [line.strip() for line in md_string.strip().split("\n") if line.strip()]
    if len(lines) < 3:
        return "<html><body><table></table></body></html>"

    html_parts = ["<html><body><table>"]

    # Process Headers (Line 0)
    html_parts.append("<thead><tr>")
    headers = [c.strip() for c in lines[0].strip("|").split("|")]
    for h in headers:
        html_parts.append(f"<td>{h}</td>")
    html_parts.append("</tr></thead>")

    # Process Body (Lines 2 onwards, skipping the separator at Line 1)
    html_parts.append("<tbody>")
    for line in lines[2:]:
        cells = [c.strip() for c in line.strip("|").split("|")]
        if not cells:
            continue
        html_parts.append("<tr>")
        for cell in cells:
            html_parts.append(f"<td>{cell}</td>")
        html_parts.append("</tr>")
    html_parts.append("</tbody>")
    html_parts.append("</table></body></html>")

    return "".join(html_parts)


# --- 2. TEDS Core Implementation ---


class TableTree(Tree):
    """Custom Tree node structure to separate HTML tags from cell content."""

    def __init__(self, tag, content=None, *children):
        self.tag = tag
        self.content = content
        super().__init__(tag, *children)


class TEDSConfig(Config):
    """
    Custom APTED configuration for TEDS.
    Penalizes structural edits by 1, and content edits by their Normalized Levenshtein distance.
    """

    def rename(self, node1, node2):
        # If both nodes are structural tags (or both are text nodes but tags differ)
        if node1.tag != node2.tag:
            return 1.0

        # If both are content nodes (text inside cells), use normalized string distance
        if node1.content is not None and node2.content is not None:
            dist = distance.levenshtein(node1.content.lower(), node2.content.lower())
            max_len = max(len(node1.content), len(node2.content))
            return dist / max_len if max_len > 0 else 0.0

        return 0.0


class TEDS:
    """Evaluates Tree Edit Distance Similarity between two HTML tables."""

    def __init__(self):
        self.config = TEDSConfig()

    def load_html_tree(self, html_str: str) -> TableTree:
        """Parses an HTML string into a tree of TableTree nodes."""
        try:
            root = html.fromstring(html_str)
            table_node = root.xpath("//table")
            if not table_node:
                return TableTree("empty")
            return self._build_tree(table_node[0])
        except Exception:
            return TableTree("empty")

    def _build_tree(self, node) -> TableTree:
        """Recursively build the tree from lxml nodes."""
        tag = node.tag.lower()
        children = []

        # If the node has text content (e.g., inside a <td>), add it as a child content node
        if node.text and node.text.strip():
            children.append(TableTree("content", content=node.text.strip()))

        for child in node:
            children.append(self._build_tree(child))
            # Handle trailing text after a child tag
            if child.tail and child.tail.strip():
                children.append(TableTree("content", content=child.tail.strip()))

        return TableTree(tag, None, *children)

    def _get_tree_size(self, node) -> int:
        """Recursively counts the number of nodes in the tree."""
        if node is None:
            return 0
        size = 1  # Count the current node
        for child in node.children:
            size += self._get_tree_size(child)
        return size

    def evaluate(self, pred_html: str, gt_html: str) -> float:
        """Calculates the TEDS score between 0.0 and 1.0."""
        if not pred_html and not gt_html:
            return 1.0
        if not pred_html or not gt_html:
            return 0.0

        tree_pred = self.load_html_tree(pred_html)
        tree_gt = self.load_html_tree(gt_html)

        # Get the size (number of nodes) of each tree using the helper
        size_pred = self._get_tree_size(tree_pred)
        size_gt = self._get_tree_size(tree_gt)

        if size_pred == 1 and size_gt == 1:
            return 0.0  # Both are empty/invalid trees

        # Compute APTED Tree Edit Distance
        apted = APTED(tree_pred, tree_gt, self.config)
        edit_distance = apted.compute_edit_distance()

        # TEDS Formula
        teds_score = 1.0 - (float(edit_distance) / max(size_pred, size_gt))
        return max(0.0, teds_score)


# --- 3. Main Pipeline Hook ---
teds_evaluator = TEDS()


def calculate_teds(pred_md: str, gt_md: str) -> float:
    """
    Main entry point for the pipeline.
    Converts markdown tables to HTML trees and computes similarity.
    """
    pred_html = markdown_to_html(pred_md)
    gt_html = markdown_to_html(gt_md)

    return teds_evaluator.evaluate(pred_html, gt_html)

Load predictions into a searchable dictionary: `preds_dict[sample_id][subfigure] = markdown_table`

In [ ]:
preds_dict = {}

with open(SUBMISSION_FILE, "r", encoding="utf-8") as f:
    submission_data = json.load(f)
    for item in submission_data:
        s_id = item.get("sample_id")
        preds_dict[s_id] = {}

        # The submission structure uses "data_extraction": {"a": "markdown", "b": "markdown"}
        for sub_key, table_md in item.get("data_extraction", {}).items():
            preds_dict[s_id][sub_key] = table_md

Parse the evaluation directory to find the corresponding ground truth markdown tables.

In [ ]:
aligned_data = {"preds": [], "gts": []}

json_files = list(EVAL_DATA_DIR.rglob("*.json"))
for json_file in json_files:
    fullpath = str(json_file)
    if ".vscode" in fullpath or "content" in fullpath:
        continue

    with open(json_file, "r", encoding="utf-8") as f:
        gt_data = json.load(f)

    sample_id = gt_data.get("sample_id")
    if not sample_id or sample_id not in preds_dict:
        continue

    # Extract tables per subfigure mapping
    for sub_key, gt_table_md in gt_data.get("data_extraction", {}).items():
        if sub_key not in preds_dict[sample_id]:
            continue

        pred_table_md = preds_dict[sample_id][sub_key]

        aligned_data["preds"].append(pred_table_md)
        aligned_data["gts"].append(gt_table_md)

Calculate the specified table extraction metrics across the dataset.

In [ ]:
print("\n🧮 Calculating Data Extraction metrics...")

teds_scores = []
rms_scores = []

for p_md, g_md in tqdm(
    zip(aligned_data["preds"], aligned_data["gts"]),
    total=len(aligned_data["preds"]),
    desc="Evaluating Tables",
):
    teds_scores.append(calculate_teds(p_md, g_md))
    rms_scores.append(calculate_rms(p_md, g_md))

results = {
    "Data Table Extraction": {
        "TEDS": np.mean(teds_scores) if teds_scores else 0.0,
        "RMS": np.mean(rms_scores) if rms_scores else 0.0,
    }
}

print("✅ Computed TEDS and RMS metrics.")

Append the evaluation metrics to the JSON file to keep a running history

In [ ]:
if RESULTS_FILE.exists():
    with open(RESULTS_FILE, "r") as f:
        try:
            history = json.load(f)
        except json.JSONDecodeError:
            history = []
else:
    history = []

history.append(
    {
        "run_type": "table_extraction_evaluation",
        "submission_file": SUBMISSION_FILE.name,
        "metrics": results,
    }
)

with open(RESULTS_FILE, "w") as f:
    json.dump(history, f, indent=4)

print(f"\n✨ Evaluation complete! State updated in: {RESULTS_FILE}")
print(json.dumps(results, indent=2))